In [4]:
# Imports
import sys
from pathlib import Path

# Resolve project root and ensure it's on sys.path
ROOT = Path.cwd().resolve()
for _ in range(5):
    if (ROOT / "pyproject.toml").exists() or (ROOT / "raw_data").exists():
        break
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils import discretize_preprocess

In [5]:
# Preprocess data
from pathlib import Path

dataset_path = ROOT / "raw_data" / "adult.csv"
output_path = ROOT / "discretized_data" / "adult.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

discretize_preprocess(str(dataset_path), str(output_path), bins=10, strategy='uniform')

Preprocessing: /home/adity/github/katabatic-mentorship-repo/raw_data/adult.csv
Saved preprocessed discrete dataset to: /home/adity/github/katabatic-mentorship-repo/discretized_data/adult.csv


In [6]:
import torch
from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.models.tabfairgdt_alex import TabFairGDT

# Device Config
device = "cuda:0" if torch.cuda.is_available() else "cpu"

# Set paths
input_csv = str(output_path)
output_dir = str(ROOT / "sample_data" / "adult")
real_test_dir = output_dir
synthetic_dir = str(ROOT / "synthetic" / "adult" / "tabfairgdt")

# set protected attribute
protected_col = input("Protected Attribute (S): ").strip()

# TabFairGDT parameters
model_config = {
    "protected_attribute": protected_col,
    "lambda_val": 1 #fairness constraint. 0 = max utility, 1 = max fairness
}

pipeline = TrainTestSplitPipeline(model=TabFairGDT)

pipeline.run(
    input_csv=input_csv,
    output_dir=output_dir,
    synthetic_dir=synthetic_dir,
    real_test_dir=real_test_dir,
    **model_config
)

Loaded data with shape: (32561, 15)
Saved train/test full data
Train size: (26048, 15), Test size: (6513, 15)
Train label distribution:
 class
0    0.759175
1    0.240825
Name: proportion, dtype: float64
Test label distribution:
 class
0    0.759251
1    0.240749
Name: proportion, dtype: float64
Saved X/y split
Training shape: (26048, 14) (26048,)
Test shape: (6513, 14) (6513,)
[TabFairGDT] Target: class | Protected: sex


/home/adity/github/katabatic-mentorship-repo/.venv/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(



Results saved to: Results/adult/tabfairgdt_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.7726
F1 Score: 0.6899
AUC: 0.8349

MLP:
Accuracy: 0.8099
F1 Score: 0.7771
AUC: 0.8561

RF:
Accuracy: 0.7847
F1 Score: 0.7603
AUC: 0.7724

XGBoost:
Accuracy: 0.7766
F1 Score: 0.7897
AUC: 0.8532


'Train test split pipeline executed successfully.'